# DermaAI — Supplementary Academic Demonstration Notebook 🩺🔬

> **ACADEMIC & PRODUCTION ISOLATION NOTICE:**  
> This Jupyter Notebook and its associated Python modules are **strictly supplementary academic demonstrations** designed for educational exploration of computer vision and medical informatics. They are **not** part of or connected to the production Next.js/Express web application, Google Gemini API, Firebase, or Vercel deployment.

## Learning Objectives
1. **Medical Imaging Preprocessing:** Understand matrix standardization ($224 \times 224$), color space conversion (BGR $\to$ RGB), and Gaussian smoothing.
2. **Signal Quality Gating:** Implement sharpness estimation via the Laplacian operator variance $\text{Var}(\nabla^2 I)$ and exposure boundary checks.
3. **Colorimetric Feature Extraction:** Decompose skin lesion imagery into the **HSV** (Hue, Saturation, Value) color space for chromatic separation.
4. **Exploratory Data Analysis (EDA):** Analyze patient scan intake modalities and confidence distributions across demographic cohorts.

---

In [ ]:
import os
import sys
from pathlib import Path

# Ensure dermai_python is in sys.path
notebook_dir = Path.cwd()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dermai_python.image_preprocessing import (
    load_image,
    resize_image,
    convert_bgr_to_rgb,
    convert_to_grayscale,
    apply_gaussian_blur,
    preprocess_pipeline,
)
from dermai_python.image_quality_check import evaluate_quality, format_quality_report
from dermai_python.image_feature_analysis import extract_all_features, format_feature_report
from dermai_python.scan_data_analysis import (
    load_scan_dataset,
    get_dataset_overview,
    compute_distributions,
    compute_grouped_statistics,
)
from dermai_python.run_demo import create_illustrative_test_image

print("All DermaAI academic modules imported successfully!")

## 1. Exploratory Data Analysis (EDA) on Scan Logs

In clinical informatics, analyzing incoming scan metadata helps monitor system performance, identify prevalent conditions, and observe model certainty across intake modalities.

In [ ]:
csv_path = Path("sample_data/sample_scans.csv")
df = load_scan_dataset(csv_path)

print(f"Loaded {len(df)} records from {csv_path.name}")
df.head()

In [ ]:
# Grouped summary statistics: Confidence by reported concern
grouped_stats = compute_grouped_statistics(df)
grouped_stats

In [ ]:
# Visualizing distributions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Reported concerns bar chart
concerns = df["reported_concern"].value_counts()
axes[0].barh(concerns.index[::-1], concerns.values[::-1], color="#2b6cb0", edgecolor="#1a365d")
axes[0].set_title("Reported Skin Concerns Distribution", fontsize=13, weight="bold")
axes[0].set_xlabel("Number of Scans")
axes[0].grid(axis="x", linestyle="--", alpha=0.5)

# 2. Input types pie chart
input_counts = df["input_type"].value_counts()
colors = ["#4299e1", "#48bb78", "#ed8936"]
axes[1].pie(
    input_counts.values,
    labels=[k.replace("_", " ").title() for k in input_counts.index],
    autopct="%1.1f%%",
    colors=colors,
    explode=[0.04] * len(input_counts),
    startangle=140,
)
axes[1].set_title("Intake Modality Distribution", fontsize=13, weight="bold")

plt.tight_layout()
plt.show()

## 2. Synthetic Skin Lesion Generation

To ensure strict privacy and compliance with healthcare data regulations (HIPAA/GDPR), we generate an illustrative synthetic dermoscopic lesion patch with realistic skin background, asymmetric pigmentation, and scale calibration markings.

In [ ]:
synthetic_patch_path = Path("demo_output/notebook_synthetic_lesion.png")
create_illustrative_test_image(synthetic_patch_path)

# Load and convert to RGB for Matplotlib visualization
bgr_patch = cv2.imread(str(synthetic_patch_path))
rgb_patch = cv2.cvtColor(bgr_patch, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(5, 5))
plt.imshow(rgb_patch)
plt.title(f"Synthetic Dermoscopic Patch ({rgb_patch.shape[1]}x{rgb_patch.shape[0]} px)", weight="bold")
plt.axis("off")
plt.show()

## 3. Computer Vision Preprocessing Pipeline

Standardizing raw images is vital for convolutional neural networks:
- **Resizing to $224 \times 224$:** Fixed matrix dimensions required by CNN architectures (ResNet, EfficientNet, MobileNet).
- **Grayscale Conversion:** Isolating the luminance gradient to evaluate edge profiles.
- **Gaussian Smoothing:** Attenuating high-frequency capture noise and surface hair artifacts.

In [ ]:
# Run complete preprocessing pipeline
prep = preprocess_pipeline(synthetic_patch_path)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(rgb_patch)
axes[0].set_title(f"1. Original\n{rgb_patch.shape[:2]}", fontsize=11, weight="bold")
axes[0].axis("off")

axes[1].imshow(prep["rgb_image"])
axes[1].set_title(f"2. Resized (224x224)\n{prep['processed_shape'][:2]}", fontsize=11, weight="bold")
axes[1].axis("off")

axes[2].imshow(prep["grayscale"], cmap="gray")
axes[2].set_title("3. Grayscale\nLuminance Channel", fontsize=11, weight="bold")
axes[2].axis("off")

axes[3].imshow(cv2.cvtColor(prep["blurred"], cv2.COLOR_BGR2RGB))
axes[3].set_title("4. Gaussian Blur\nNoise Suppressed", fontsize=11, weight="bold")
axes[3].axis("off")

plt.tight_layout()
plt.show()

## 4. Optical Quality Verification & Blur Detection

Before clinical images are analyzed, an optical gate verifies whether the image is sharp and well-exposed.

$$\text{Sharpness Score} = \text{Var}\left(\nabla^2 I\right) = \text{Var}\left(\frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}\right)$$

In [ ]:
quality_results = evaluate_quality(prep["resized_bgr"])
print(format_quality_report(quality_results))

## 5. Descriptive Colorimetric Feature Extraction (HSV Space)

The **HSV (Hue, Saturation, Value)** color space separates chromatic information from illumination:
- **Hue:** Reflects the dominant spectral wavelength (differentiating erythema/redness from melanin/brown pigment).
- **Saturation:** Color intensity / blood pooling concentration.
- **Value:** Surface brightness.

In [ ]:
features = extract_all_features(prep["resized_bgr"])
print(format_feature_report(features))

# Plot HSV Channel Histograms
hsv = cv2.cvtColor(prep["resized_bgr"], cv2.COLOR_BGR2HSV)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = ["Hue Distribution (0-179)", "Saturation Distribution (0-255)", "Value/Brightness Distribution (0-255)"]
colors = ["#e53e3e", "#dd6b20", "#3182ce"]

for i in range(3):
    axes[i].hist(hsv[:, :, i].ravel(), bins=30, color=colors[i], edgecolor="black", alpha=0.75)
    axes[i].set_title(titles[i], weight="bold")
    axes[i].set_ylabel("Pixel Count")
    axes[i].grid(linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 6. Academic Conclusion & Medical Disclaimer

> **IMPORTANT MEDICAL & SCIENTIFIC DISCLAIMER:**  
> The algorithms and statistical features demonstrated in this notebook are generic computer vision techniques. They do **not** assess clinical malignancy, melanoma risk, or histological condition. They must **never** be used for patient diagnosis or clinical decision-making. Always consult a board-certified dermatologist for medical evaluations.